data cleaning pipeline


In [ ]:
import pandas as pd
import numpy as np
import json


In [ ]:
df = pd.read_csv('../data/raw/nyc_parking_tickets_sample.csv', dtype=str)
df.columns = df.columns.str.replace(' ', '_')


In [ ]:
df['Issue_Date'] = pd.to_datetime(df['Issue_Date'], format='%m/%d/%Y', errors='coerce')


In [ ]:
df['Fine_Amount'] = pd.to_numeric(df['Fine_Amount'], errors='coerce')
df.loc[df['Fine_Amount'] <= 0, 'Fine_Amount'] = np.nan
df['Fine_Amount'] = df['Fine_Amount'].fillna(df.groupby('Violation_Description')['Fine_Amount'].transform('median'))


In [ ]:
df['Vehicle_Year'] = pd.to_numeric(df['Vehicle_Year'], errors='coerce')
df.loc[(df['Vehicle_Year'] == 9999) | (df['Vehicle_Year'] > 2014) | (df['Vehicle_Year'] < 1980), 'Vehicle_Year'] = np.nan


In [ ]:
color_map = {'GY': 'GRAY', 'BK': 'BLACK', 'WH': 'WHITE'}
df['Vehicle_Color'] = df['Vehicle_Color'].str.upper().replace(color_map)


In [ ]:
df['Violation_County'] = df['Violation_County'].str.title()


In [ ]:
top_makes = df['Vehicle_Make'].value_counts().nlargest(15).index
df['Vehicle_Make'] = np.where(df['Vehicle_Make'].isin(top_makes), df['Vehicle_Make'], 'OTHER')


In [ ]:
passenger = ['SUBN', '4DSD', '2DSD']
commercial = ['VAN', 'DELV', 'PICK']
df['Vehicle_Type_Group'] = 'OTHER'
df.loc[df['Vehicle_Body_Type'].isin(passenger), 'Vehicle_Type_Group'] = 'Passenger'
df.loc[df['Vehicle_Body_Type'].isin(commercial), 'Vehicle_Type_Group'] = 'Commercial'


In [ ]:
valid_states = ['NY', 'NJ', 'PA', 'CT', 'FL', 'MA', 'TX', 'VA', 'MD', 'NC']
df.loc[~df['Registration_State'].isin(valid_states), 'Registration_State'] = 'UNKNOWN'
df['Is_OutOfState'] = (df['Registration_State'] != 'NY').astype(int)


In [ ]:
df['Street_Name'] = df['Street_Name'].str.upper().str.strip()


In [ ]:
plate_counts = df['Plate_ID'].value_counts()
df['Is_Repeat_Offender'] = df['Plate_ID'].map(plate_counts > 1).astype(int)


In [ ]:
df = df.drop_duplicates(subset=['Summons_Number'])


In [ ]:
transformation_log = {
    "step_01_dates": "3 formats -> unified YYYY-MM-DD, NaT for unparseable",
    "step_02_fines": "Negatives and zeros -> NaN, imputed from violation median",
    "step_03_years": "9999 and >2014 or <1980 -> NaN",
    "step_04_colors": "GY->GRAY, BK->BLACK, WH->WHITE, mixed case -> UPPER",
    "step_05_boroughs": "Mixed case -> Title Case, abbreviations mapped",
    "step_06_makes": "top 15 kept, rest -> OTHER",
    "step_07_body_type": "10 codes -> 4 groups",
    "step_08_states": "Invalid codes -> UNKNOWN, Is_OutOfState flag added",
    "step_09_streets": "UPPER + strip",
    "step_10_plates": "Ticket count per plate, repeat offender flags added",
    "step_11_dedup": "Duplicate Summons_Number removed",
    "step_12_validate": "All assertions passed"
}
with open('../docs/etl_transformation_log.json', 'w') as f:
    json.dump(transformation_log, f, indent=2)


In [ ]:
df.to_csv('../data/processed/nyc_parking_clean.csv', index=False)
